# Correlation 
## Introduction


## Definitions
### Correlation
### Types of correlation
### Bland-Altman


## Descriptive statistics and visualization


In [ ]:
import pandas as pd

data_borkman = pd.DataFrame({
    'per_C2022_fatacids': [  # percentage of C20-22 fatty acids
        17.9, 18.3, 18.3, 18.4, 18.4, 20.2, 20.3, 21.8, 
        21.9, 22.1, 23.1, 24.2, 24.4],
    'insulin_sensitivity': [  # insulin sensitivity index
        250, 220, 145, 115, 230, 200, 330, 400, 370, 260, 270, 530, 375]})

print("First row of the Borkman dataset:")
print(data_borkman.head())

### Descriptive Statistics


In [ ]:
print("Descriptive statistics using Pandas:")
print(data_borkman.describe().round(4))

In [ ]:
from scipy.stats import describe as scipy_describe

print("Descriptive statistics using SciPy:")
# We use a combination of transformation of the results to dictionaries
# with concatenation to enhance the output
print(
    pd.concat([
        pd.Series(
        scipy_describe(data_borkman['per_C2022_fatacids'])._asdict(),
        name='per_C2022_fatacids'),
        pd.Series(
            scipy_describe(data_borkman['insulin_sensitivity'])._asdict(),
            name='insulin_sensitivity'),
    ], axis=1))

### Visualization
#### Scatter plots


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(3.5, 3))
plt.scatter(
    data_borkman['per_C2022_fatacids'],
    data_borkman['insulin_sensitivity'])

plt.title('Correlation analysis')
plt.xlabel('%C20-22 fatty acids')
plt.ylabel('Insulin sensitivity (mg/m²/min)');

#### Correlation matrices and heatmaps


In [ ]:
import pingouin as pg

# Load the 'penguins' dataset
penguins = pg.read_dataset('penguins')

print("First rows of the Palmer penguins dataset:")
print(penguins.head())

In [ ]:

# Calculate the correlation matrix (excluding non-numeric columns)
corr_matrix = penguins[
    ['bill_length_mm', 'bill_depth_mm', 
     'flipper_length_mm', 'body_mass_g']].corr()

# Create a heatmap
plt.figure(figsize=(3.5, 3))
sns.heatmap(corr_matrix, annot=True, cmap='vlag')
plt.title('Correlation matrix heatmap');

## Assumptions of correlation analysis
### Linearity


In [ ]:

plt.figure(figsize=(3.5, 3))

# Fit a linear regression model using Pingouin
model = pg.linear_regression(
    data_borkman['per_C2022_fatacids'],
    data_borkman['insulin_sensitivity'])

# Extract the residuals using the residuals_ attribute
residuals = model.residuals_

# Create a residual plot
sns.residplot(
    x=data_borkman['per_C2022_fatacids'],
    y=residuals,
    lowess=True,  # Adds a smooth curve to visualize the trend of residuals
    line_kws=dict(color="r"))
plt.title('Residual plot')
plt.xlabel('%C20-22')
plt.ylabel('Residuals');

### Normality


In [ ]:
import warnings
warnings.filterwarnings('ignore')

print("Normality tests in correlation (Borkman dataset)")
print("-"*48)
print("D'Agostino-Pearson:")
# Tests all variables when giving a wide-format dataframe
print(pg.normality(data_borkman, method='normaltest'))
print()  # Print blank separation line
print("Shapiro-Wilk:")
print(pg.normality(data_borkman, method='shapiro'))

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(6, 3))

for i, column_name in enumerate(data_borkman.columns):
    pg.qqplot(
        data_borkman[column_name],
        ax=axes[i],)
    axes[i].set_title(column_name)

plt.tight_layout();

### Homoscedasticity


In [ ]:
# Calculate the median cutoff for %C20-22
cutoff = data_borkman['per_C2022_fatacids'].median()

# Create boolean masks for low and high %C20-22 groups
ix_lo = data_borkman['per_C2022_fatacids'] <= cutoff
ix_hi = data_borkman['per_C2022_fatacids'] > cutoff

# Extract insulin sensitivity values for each group using .loc
low_fatacids = data_borkman.loc[ix_lo, 'insulin_sensitivity'].to_numpy()
high_fatacids = data_borkman.loc[ix_hi, 'insulin_sensitivity'].to_numpy()

# Perform Levene's test for homoscedasticity
print("Levene's test for homoscedasticity between low and high %C20-22:")
print(pg.homoscedasticity([low_fatacids, high_fatacids], method='levene'))

### Outliers


In [ ]:

# Create separate boxplots for each variable to respect each scale
_, axes = plt.subplots(nrows=1, ncols=2, figsize=(6, 3))

sns.stripplot(
    y=data_borkman['per_C2022_fatacids'],
    color='dodgerblue', size=8, alpha=.8,
    ax=axes[0],)
sns.boxplot(
    y=data_borkman['per_C2022_fatacids'],
    width=.3,
    boxprops={'facecolor': 'none'},  # Make the boxplot transparent
    medianprops={'color': 'black'},
    ax=axes[0])

sns.stripplot(
    y=data_borkman['insulin_sensitivity'],
    color='crimson', size=8, alpha=.8,
    ax=axes[1],)
sns.boxplot(
    y=data_borkman['insulin_sensitivity'],
    width=.3,
    boxprops={'facecolor': 'none'},
    medianprops={'color': 'black'},
    ax=axes[1])

plt.xticks()
plt.tight_layout()
sns.despine();

## Measures of correlation
### Covariance


### Covariance matrix


In [ ]:
# Calculate the sample covariance matrix with Pandas
covariance_matrix_pandas = data_borkman.cov(ddof=1)

print("Sample covariance matrix of the Borkman dataset:")
print(covariance_matrix_pandas.round(4))

In [ ]:
import numpy as np

# Calculate the covariance using NumPy
covariance_matrix_numpy = np.cov(
    data_borkman['per_C2022_fatacids'],
    data_borkman['insulin_sensitivity'],
    ddof=1)
covariance_numpy = covariance_matrix_numpy[0, 1]

print(
    f"Covariance between %C20-22 and insulin sensitivity = \
{covariance_numpy:.3f}")

### Pearson correlation coefficient
#### How the Pearson correlation coefficient is calculated


In [ ]:
def pearson_r_custom(x, y):
    """
    Computes the Pearson correlation coefficient between two numpy arrays,
    illustrating the step-by-step calculation.

    Args:
        x: the first numpy array of data points.
        y: the second numpy array of data points.

    Returns:
        float: the Pearson correlation coefficient (r) between x and y.

    Raises:
        ValueError: if the input arrays have different lengths.
    """
    if len(x) != len(y):
        raise ValueError("Input arrays x and y must have the same length.")

    n = len(x)  # Get the number of data points

    # 1. Calculate the means of X and Y (center of gravity)
    mean_x = np.mean(x)
    mean_y = np.mean(y)

    # 2. Calculate the distances to the center of gravity
    dist_x = x - mean_x
    dist_y = y - mean_y

    # 3. Standardize the dstances
    std_x = np.std(x, ddof=1)  # Use ddof=1 for sample standard deviation
    std_y = np.std(y, ddof=1)
    std_dist_x = dist_x / std_x
    std_dist_y = dist_y / std_y

    # 4. Multiply the standardized distances
    products = std_dist_x * std_dist_y

    # 5. Sum the products
    sum_products = np.sum(products)

    # 6. Divide by n-1
    r = sum_products / (n - 1)

    return r

In [ ]:
# Calculate the Pearson coefficient of correlation between %C20-22
# and insulin sensitivity using the previous custom function
r_custom = pearson_r_custom(
    x=data_borkman['per_C2022_fatacids'],
    y=data_borkman['insulin_sensitivity'])

# Calculate Pearson r as s_xy / (s_x * s_y)
r_product = (
    covariance_numpy
    / (
        data_borkman['per_C2022_fatacids'].std()
        #same as covariance_matrix_pandas.iloc[0, 0]**.5
        * data_borkman['insulin_sensitivity'].std()))
        #same as covariance_matrix_pandas.iloc[1, 1]**.5

# Print all results
print(f"Pearson correlation coefficient (custom function) = {r_custom:.3f}")
print(f"Pearson r (covariance divided by product of standard deviations) = \
{r_product:.3f}")

#### Computing Pearson's r with Python


In [ ]:
from scipy.stats import pearsonr

# Calculate Pearson r using NumPy
pearson_r_numpy = np.corrcoef(
    data_borkman['per_C2022_fatacids'],
    data_borkman['insulin_sensitivity'])[0,1]

# Extract Pearson r from correlation matrix using Pandas
pearson_r_pandas = data_borkman.corr().iloc[0, 1]

# Calculate Pearson correlation coefficient and the associated P value
pearson_r_result_scipy = pearsonr(
    data_borkman['per_C2022_fatacids'], data_borkman['insulin_sensitivity'])
# Extract the statistics with tuple unpacking
pearson_r_scipy, pearson_p_scipy = pearson_r_result_scipy
# Print all the results
print(f"Pearson r (NumPy) = {pearson_r_numpy:.5f}")
print(f"Pearson r (Pandas)= {pearson_r_pandas:.5f}")
print(f"Pearson r (SciPy) = {pearson_r_scipy:.5f}")

#### The r² coefficient of determination


In [ ]:
print(f"r² coefficient of determination = {r_custom**2:.2f}")

### Non-parametric methods 
#### Spearman rank correlation coefficient


In [ ]:
def spearman_rho_custom(x, y):
    """
    Computes Spearman's rank correlation coefficient (rho) between 
    two numpy arrays, illustrating the step-by-step calculation.

    Args:
        x: the first numpy array of data points.
        y: the second numpy array of data points.

    Returns:
        float: Spearman's rank correlation coefficient (rho) between x and y.

    Raises:
        ValueError: if the input arrays have different lengths.
    """
    if len(x) != len(y):
        raise ValueError("Input arrays x and y must have the same length.")

    n = len(x)

    # 1. Rank the data using numpy.argsort (get index of the sorted array)
    rank_x = np.argsort(x)
    rank_y = np.argsort(y)

    # 2. Calculate the difference in ranks (di) - array
    d = rank_x - rank_y

    # 3. Square the differences (di^2)
    d_squared = d**2

    # 4. Sum the squared differences
    sum_d_squared = np.sum(d_squared)

    # 5. Apply the Spearman formula
    rho = 1 - (6 * sum_d_squared) / (n * (n**2 - 1))

    return rho

In [ ]:
# Calculate the Spearman rho coefficient of correlation between %C20-22
# and insulin sensitivity using the previous custom function
rho_custom = spearman_rho_custom(
    x=data_borkman['per_C2022_fatacids'],
    y=data_borkman['insulin_sensitivity'])

print(f"Spearman ρ correlation coefficient (custom function) = \
{rho_custom:.3f}")

#### Kendall rank correlation coefficient


In [ ]:
def kendall_tau_custom(x, y):
    """
    Computes Kendall's tau correlation coefficient between two numpy arrays,
    illustrating the step-by-step calculation.

    Args:
        x: the first numpy array of data points.
        y: the second numpy array of data points.

    Returns:
        float: Kendall's tau correlation coefficient between x and y.

    Raises:
        ValueError: if the input arrays have different lengths.
    """
    if len(x) != len(y):
        raise ValueError("Input arrays x and y must have the same length.")

    n = len(x)

    # Calculate the number of concordant and discordant pairs
    concordant: int = 0
    discordant: int = 0

    for i in range(n):
        for j in range(i + 1, n):  # Iterate through all unique pairs
            if (x[i]>x[j] and y[i]>y[j]) or (x[i]<x[j] and y[i]<y[j]):
                concordant += 1
            elif (x[i]>x[j] and y[i]<y[j]) or (x[i]<x[j] and y[i]>y[j]):
                discordant += 1

    # Calculate Kendall's tau
    tau = (concordant - discordant) / (n * (n - 1) / 2)

    return tau

In [ ]:
# Calculate the Kendall tau coefficient of correlation between %C20-22
# and insulin sensitivity using the previous custom function
tau_custom = kendall_tau_custom(
    x=data_borkman['per_C2022_fatacids'],
    y=data_borkman['insulin_sensitivity'])

print(f"Kendall τ correlation coefficient (custom function) = \
{tau_custom:.3f}")

#### Calculating Spearman ρ and Kendall τ with Python


In [ ]:
from scipy.stats import spearmanr

# Calculate Spearman rho using SciPy
spearman_rho_scipy, spearman_p_scipy = spearmanr(
    data_borkman['per_C2022_fatacids'], data_borkman['insulin_sensitivity'])

# Extract Spearman rho from correlation matrix using Pandas
spearman_rho_pandas = data_borkman.corr(method='spearman').iloc[0, 1]

# Print all the results
print(f"Spearman ρ (SciPy) = {spearman_rho_scipy:.3f}")
print(f"Spearman ρ (Pandas)= {spearman_rho_pandas:.3f}")

In [ ]:
from scipy.stats import kendalltau

# Calculate Kendall tau using SciPy
kendall_tau_scipy, kendall_p_scipy = kendalltau(
    data_borkman['per_C2022_fatacids'], data_borkman['insulin_sensitivity'])

# Extract Kendall tau from correlation matrix using Pandas
kendall_tau_pandas = data_borkman.corr(method='kendall').iloc[0, 1]

# Print all the results
print(f"Kendall τ (SciPy) = {kendall_tau_scipy:.3f}")
print(f"Kendall τ (Pandas)= {kendall_tau_pandas:.3f}")

#### Handling ties


In [ ]:
# Calculate Spearman rho using np.corrcoef and pd.Series.rank
# with method='first'; same results as `spearman_rho_custom`
spearman_numpy_ordinal = np.corrcoef(
    data_borkman['per_C2022_fatacids'].rank(method='first'),
    data_borkman['insulin_sensitivity'].rank(method='first'))[0, 1]

# Calculate Spearman rho using np.corrcoef and pd.Series.rank
# with method='average' (default); same results as `spearmanr`
spearman_numpy_average = np.corrcoef(
    data_borkman['per_C2022_fatacids'].rank(method='average'),
    data_borkman['insulin_sensitivity'].rank(method='average'))[0, 1]

# Print all results
print(f"Spearman ρ (NumPy and ordinal rank) = {spearman_numpy_ordinal:.3f}")
print(f"Spearman ρ (NumPy and average rank) = {spearman_numpy_average:.3f}")

In [ ]:
def kendall_tau_b_custom(x, y):
    """
    Computes Kendall's tau-b correlation coefficient between two numpy arrays,
    illustrating the step-by-step calculation with explicit tie handling.
    Args:
        x: the first numpy array of data points.
        y: the second numpy array of data points.

    Returns:
        float: Kendall's tau-b correlation coefficient between x and y.

    Raises:
        ValueError: if the input arrays have different lengths.
    """
    if len(x) != len(y):
        raise ValueError("Input arrays x and y must have the same length.")

    n = len(x)

    # Initialize counters for concordant, discordant, and tied pairs
    concordant: int = 0
    discordant: int = 0
    ties_x: int = 0  # Number of tied pairs in x
    ties_y: int = 0  # Number of tied pairs in y

    # Iterate through all unique pairs of data points
    for i in range(n):
        for j in range(i + 1, n):
            # Check for concordant pairs (both x and y increase 
            # or decrease together)
            if (x[i]>x[j] and y[i]>y[j]) or (x[i]<x[j] and y[i]<y[j]):
                concordant += 1
            # Check for discordant pairs (x increases, y decreases, 
            # or vice versa)
            elif (x[i]>x[j] and y[i]<y[j]) or (x[i]<x[j] and y[i]>y[j]):
                discordant += 1
            # Check for ties in x
            elif x[i] == x[j]:
                ties_x += 1
            # Check for ties in y
            elif y[i] == y[j]:
                ties_y += 1

    # Calculate Kendall's tau-b using the adjusted formula for ties
    tau_b = (concordant - discordant) / np.sqrt(
        (concordant + discordant + ties_x)
        * (concordant + discordant + ties_y))

    return tau_b

In [ ]:
# Calculate the Kendall tau-b coefficient of correlation 
# between %C20-22 and insulin sensitivity using the 
# previous custom function; same as SciPy `kendalltau`
tau_b_custom = kendall_tau_b_custom(
    x=data_borkman['per_C2022_fatacids'],
    y=data_borkman['insulin_sensitivity'])

print(f"Kendall τ-b correlation coefficient (custom function) = \
{tau_b_custom:.3f}")

### Exploration of nonlinear patterns


In [ ]:

σ = 1.0 / 5

# Generate data for each type of relationship
linear = [[u, (u)/100 + σ*np.random.randn()] for u in range(10, 500)]
monotonic = [
    [u, 50*(0.8**(u/10)) + σ*np.random.randn()] for u in range(10, 300)]
non_monotonic = [
    [u, (u)**3+3*u**2 + σ*np.random.randn()] for u in np.arange(
        -1.5, 1.5,1.0 / 250)]

# Combine the distribution with a title string
data = {
    "Linear": linear,
    "Monotonic": monotonic,
    "Non-monotonic": non_monotonic}

# Create subplots
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(6, 2.25))

# Plot each relationship
for i, (title, model) in enumerate(data.items()):
    x = [point[0] for point in model]
    y = [point[1] for point in model]

    axes[i].scatter(x, y, marker='.')

    # Calculate and display correlation coefficients
    pearson_corr = pearsonr(x, y)[0]
    spearman_corr = spearmanr(x, y)[0]
    kendall_corr = kendalltau(x, y)[0]

    axes[i].text(
        x=.1, y=.95,
        s=f"Pearson: {pearson_corr:.3f}\nSpearman: \
{spearman_corr:.3f}\nKendall: {kendall_corr:.3f}",
        fontsize=8,
        va='top',
        transform=axes[i].transAxes  # Specifies the axis x,y coordinates
    )
    axes[i].set_title(title)

plt.tight_layout();

## Statistical inference
### Testing for non-zero correlation
#### Calculating the t-statistic for correlation


In [ ]:
# Calculate the t-statistic manually
df_correlation = len(data_borkman) - 2
t_correlation_manual = (
    pearson_r_scipy * np.sqrt(df_correlation / (1 - pearson_r_scipy**2)))

# Print the t-statistic
print(f"t-statistic for correlation (manual) = {t_correlation_manual:.3f} \
with {df_correlation} degrees of freedom")

#### P value


In [ ]:
from scipy.stats import t as t_dist  # Good practice to rename the t object

# Calculate the P value using the t-distribution (two-sided test)
p_value_correlation = 2 * (
    1 - t_dist(df=df_correlation).cdf(abs(t_correlation_manual)))

# Print the results
print(f"P value for the correlation t-test = {p_value_correlation:.6f}")

#### Visualizing t, critical, and P values


In [ ]:

# Significance level (alpha)
α = 0.05

# Calculate critical t-values (two-tailed test)
t_crit_lower = t_dist.ppf(α / 2, df_correlation)
t_crit_upper = t_dist.ppf(1 - α / 2, df_correlation)

# Generate x values for plotting
x_t = np.linspace(-5, 5, 1000)
hx_t = t_dist.pdf(x_t, df_correlation)

# Plot the t-distribution
plt.figure(figsize=(3.5, 3))
plt.plot(x_t, hx_t, lw=2, color='black')

# Plot the critical t-values
plt.axvline(
    x=t_crit_lower,
    color='orangered', linestyle='--',)

plt.axvline(
    x=t_crit_upper,
    color='orangered', linestyle='--',
    label=f"t* ({t_crit_upper:.2f})")

# Shade the rejection regions (alpha)
plt.fill_between(
    x_t[x_t <= t_crit_lower], hx_t[x_t <= t_crit_lower],
    linestyle="-", linewidth=2, color='tomato',
    label=f"α ({α})")
plt.fill_between(
    x_t[x_t >= t_crit_upper], hx_t[x_t >= t_crit_upper],
    linestyle="-", linewidth=2, color='tomato')

# Plot the observed t-statistic
plt.axvline(
    x=t_correlation_manual,
    color='limegreen', linestyle='-.',
    label=f"t ({t_correlation_manual:.2f})")

# Shade the P value areas (two-tailed)
plt.fill_between(
    x_t[x_t <= -abs(t_correlation_manual)],
    hx_t[x_t <= -abs(t_correlation_manual)],
    color='greenyellow',
    label=f"P ({p_value_correlation:.3f})")
plt.fill_between(
    x_t[x_t >= abs(t_correlation_manual)],
    hx_t[x_t >= abs(t_correlation_manual)],
    color='greenyellow')

# Add labels and title
plt.xlabel(r"t")
plt.ylabel('Density')
plt.margins(x=0, y=0)
plt.yticks([])
plt.title(f"t-distribution (DF={df_correlation})")
plt.legend();

#### Effect size


### Confidence interval


In [ ]:
from scipy.stats import norm

# Fisher's r-to-z transformation
z_fisher = np.arctanh(pearson_r_scipy)
# same as
#z_fisher = 0.5 * np.log((1 + pearson_r_scipy) / (1 - pearson_r_scipy))

se_r_zspace = 1 / np.sqrt(len(data_borkman) - 3)

# Calculate the confidence interval
z_critical = norm.ppf((1 + (1 - α)) / 2)
ci_r_zspace = np.array([  # CI in z-space
    z_fisher - z_critical * se_r_zspace,
    z_fisher + z_critical * se_r_zspace])

# Back-transform to r-space
ci_r_rspace = np.tanh(ci_r_zspace)
# same as
#ci_r_rspace = (np.exp(2 * ci_r_zspace)-1) / (np.exp(2 * ci_r_zspace)+1)

# Print the results
print(
    "95% CI for Pearson correlation coefficient:",
    ci_r_rspace.round(5))

### Performing statistical inference in Python


In [ ]:
# Calculate parametric confidence interval using SciPy
ci_r_scipy = np.array(pearson_r_result_scipy.confidence_interval())

print("95% CI for Pearson r (SciPy):", ci_r_scipy.round(5))

In [ ]:
# Calculate confidence interval using Pingouin
ci_r_pingouin = pg.compute_esci(
        stat=pearson_r_scipy,
        nx=len(data_borkman),
        ny=len(data_borkman),
        eftype='r',
        decimals=5,)

print("95% CI for Pearson r (`pg.compute_esci`):", ci_r_pingouin)

In [ ]:
# Calculate Pearson correlation, P value and CI using Pingouin
print("Pearson correlation inference (Pingouin):")
print(
    pg.corr(
        x=data_borkman['per_C2022_fatacids'],
        y=data_borkman['insulin_sensitivity'],
        method='pearson',  # default
        alternative='two-sided',  # default
    ).round(5))

In [ ]:
# Calculate Spearman correlation, P value and CI using Pingouin
print("Spearman correlation one-sided inference (Pingouin):")
print(
    pg.corr(
        x=data_borkman['per_C2022_fatacids'],
        y=data_borkman['insulin_sensitivity'],
        method='spearman',
        alternative='greater'))

## Bootstrapping and permutation tests for correlation analysis
### Generating bootstrap paired data


In [ ]:
def draw_bs_pairs(x, y, func=pearsonr, size=1):
    """
    Perform pairs bootstrap for correlation and compute single statistics 
    from the replicates.

    This function performs pairs bootstrap resampling for correlation 
    analysis. It takes two arrays (x and y) and a function (func) as input. 
    It resamples pairs of data points from x and y with replacement and 
    applies the provided function to calculate a statistic (e.g., 
    correlation coefficient) for each bootstrap sample.

    Args:
      x: the first data array.
      y: the second data array.
      func: the function from scipy.stats to apply to the resampled data 
      (default: pearsonr).
      size: the number of bootstrap replicates to generate (default: 1).

    Returns:
      An array of bootstrap replicates of the statistic.
    """
    # Set up array of indices to sample from
    inds = np.arange(len(x))

    # Initialize array to store bootstrap replicates
    bs_replicates = np.empty(size)

    # Generate replicates
    for i in range(size):
        # Resample indices with replacement
        bs_inds = np.random.choice(inds, len(inds), replace=True)

        # Get the resampled data (go by pairs)
        bs_x, bs_y = x[bs_inds], y[bs_inds]

        # Calculate the statistic using the provided function
        # and extract the correlation coefficient
        bs_replicates[i], _ = func(bs_x, bs_y)  # Extract the correlation coefficient

    return bs_replicates

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Set the number of replicates
B = 10_000

# Generate bootstrap replicates of Pearson's r
bs_replicates_correlation = draw_bs_pairs(
    data_borkman['per_C2022_fatacids'].to_numpy(),  # Make sure we use array
    data_borkman['insulin_sensitivity'].to_numpy(),
    func=pearsonr,
    size=B)

print("First ten calculated bootstrap Pearson correlation coefficient:")
print(bs_replicates_correlation[:10].round(3))

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Extract arrays
x=data_borkman['per_C2022_fatacids'].to_numpy()
y=data_borkman['insulin_sensitivity'].to_numpy()
inds=np.arange(len(x))

bs_replicates_correlation_b = np.array([
    pearsonr(
    x[bs_inds], y[bs_inds]).statistic # Extract the Pearson r coefficient
    for bs_inds in np.random.choice(
        inds, size=(B, len(inds)), replace=True)])

print("First ten calculated bootstrap Pearson r (pythonic):")
print(bs_replicates_correlation_b[:10].round(3))

### Estimate of the confidence interval


In [ ]:
# Calculate the 95% percentile interval using np.percentile
bs_r = np.mean(bs_replicates_correlation)
bs_ci = np.percentile(bs_replicates_correlation, [2.5, 97.5])

# Print the results
print(f"Bootstrap estimate of Pearson r = {bs_r:.3f}")
print("95% bootstrap percentile interval: ", bs_ci.round(3))

In [ ]:

plt.figure(figsize=(3.5, 3))

# histogram of the bootstrap distribution of correlation coefficients
plt.hist(
    bs_replicates_correlation,
    density=False, bins='auto',
    color='gold',
    label='Bootstrap r')

# Annotate the observed Pearson's correlation
plt.axvline(
    x=pearson_r_scipy,
    color='cornflowerblue', linestyle='-', lw=2,
    label=f'Observed r ({pearson_r_scipy:.3f})')

# Add lines for the confidence interval
plt.axvline(
    x=bs_ci[0],
    color='red', linestyle='--',
    label=f'2.5th ({bs_ci[0]:.3f})')
plt.axvline(
    x=bs_ci[1],
    color='red', linestyle='--',
    label=f'97.5th ({bs_ci[1]:.3f})')

# Add labels and title
plt.xlabel('r')
plt.ylabel('Frequency')
plt.title(f"Bootstrap Pearson r")
plt.legend();

In [ ]:
bs_ci_r_to_z = np.tanh(  # z-to-r back-transformation
    pg.compute_bootci(
        x=data_borkman['per_C2022_fatacids'],
        y=data_borkman['insulin_sensitivity'],
        paired=True,  # Resample the pairs (x_i, y_i)
        func=lambda x,y: np.arctanh(pearsonr(x, y).statistic),
        # r-to-z transformation
        confidence=0.95,
        method='per',
        seed=111,
        n_boot=10000,
        return_dist=False))

print(
    "Bootstrap 95% percentile interval (r-to-z transformation; Pingouin):",\
bs_ci_r_to_z.round(3))

In [ ]:
from scipy.stats import BootstrapMethod

# Configure the method of computation of the bootstrap CI
method = BootstrapMethod(method='BCa', random_state=111)

ci_bca_scipy = np.array(
    pearson_r_result_scipy.confidence_interval(method=method))

print("BCa 95% confidence interval (SciPy): ", ci_bca_scipy.round(3))

### P value using permutation


In [ ]:
def permute_for_correlation(x, y):
    """
    Generates a permuted sample for correlation analysis 
    under the null hypothesis of no association.

    Args:
      x: the first data array.
      y: the second data array.

    Returns:
      A new array with the values of x permuted randomly.
    """
    # Permute the values of x
    permuted_x = np.random.permutation(x)

    return permuted_x, y  # Return the permuted x and the original y

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Generate permutation replicates of the correlation coefficient
B = 10**4 - 1
permuted_r = np.empty(B)

# Generate permuted correlation coefficients
for i in range(B):
    # Permute the data
    permuted_x, y = permute_for_correlation(
        data_borkman['per_C2022_fatacids'],
        data_borkman['insulin_sensitivity'])

    # Calculate the correlation coefficient for this permuted sample
    permuted_r[i] = pearsonr(permuted_x, y).statistic

# More pythonic:
# permut_corrs = np.array([
#     pearsonr(np.random.permutation(
#         data_borkman['per_C2022_fatacids']),
#         data_borkman['insulin_sensitivity']).statistic
#     for _ in range(B)])

print("First ten calculated permuted Pearson r:")
print(permuted_r[:10].round(3))

#### One-tailed permuted P value for correlation


In [ ]:
# Calculate one-sided P value using permuation, considering the direction 
# of the observed Pearson correlation coefficient
# 1. Calculate lower and upper tail
if pearson_r_scipy >= 0:
    p_value_permuted_up = np.mean(permuted_r >= pearson_r_scipy)
    p_value_permuted_lo = np.mean(permuted_r <= -pearson_r_scipy)
else:
    p_value_permuted_lo = np.mean(permuted_r <= pearson_r_scipy)
    p_value_permuted_up = np.mean(permuted_r >= -pearson_r_scipy)

# 2. Attribute the one-tailed P value
p_value_permuted_1t = p_value_permuted_up if pearson_r_scipy >= 0 \
else p_value_permuted_lo

# Print the P value
print(f"One-sided P value obtained using permutation = \
{p_value_permuted_1t:.4f}")

In [ ]:

import matplotlib.patches as mpatches

plt.figure(figsize=(3.5, 3))

# Plot the histogram of the Pearson r of shuffled pairs
hist, bins, patches = plt.hist(
    permuted_r,
    density=False, bins=int(B**.5),
    color='gold',)

# Annotate the observed Pearson r
plt.axvline(
    x=pearson_r_scipy,
    color='limegreen', linestyle='-.', lw=2,
    label=f"Observed r ({pearson_r_scipy:.3f})")

# Determine the direction of the observed difference and plot accordingly
if pearson_r_scipy >= 0:
    # Plot the histogram of the Pearson rs >= observed Pearson r 
    extreme_diffs = permuted_r[permuted_r >= pearson_r_scipy]
    # Change the color of the bars based on the direction parameter
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge >= extreme_diffs):
            patches[i].set_facecolor('lime')
else:
    # Plot the histogram of the Pearson rs <= observed Pearson r
    extreme_diffs = permuted_r[permuted_r <= pearson_r_scipy]
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge <= extreme_diffs):
            patches[i].set_facecolor('lime')

# Add labels and title
plt.xlabel('r')
plt.ylabel('Frequency')
plt.title(f"Permuted Pearson r under H0")

# Create a copy of the original patch for the legend
original_patch = mpatches.Patch(color='gold', label='Permuted r')
# Create a patch for the legend
p_value_patch = mpatches.Patch(
    color='lime', label=fr'One-tailed P ({p_value_permuted_1t:.4f})')

# Add the patches to the legend
plt.legend(handles=[original_patch, plt.gca().lines[0], p_value_patch]);

#### Estimating two-tailed permuted P value


In [ ]:
# Maximum one-tail
p_value_permuted_2t_max = 2 * max(p_value_permuted_lo, p_value_permuted_up)

# Sum of the tails
p_value_permuted_2t_sum = p_value_permuted_lo + p_value_permuted_up

# Print the results
print(f"Two-tailed permutation P value (conservative) = \
{p_value_permuted_2t_max:.4f}")
print(f"Two-tailed permutation P value (both tails) = \
{p_value_permuted_2t_sum:.4f}")

## Conclusion
